# Oversampled channelizer

`PolyphaseAnalysisChannelizer` is critically sampled by default: it decimates by `M` (the number of
channels), so each channel's output is sampled at exactly the Nyquist rate for its own *nominal*
bandwidth `Fs/M`. The prototype filter's transition band isn't a brick wall though, so a signal sitting
near the crossover between two adjacent channels has energy that spills past `Fs/(2M)` -- and because the
channel is sampled right at that rate, that spillover **aliases back on top of the channel's own passband**
instead of showing up at a distinguishable frequency. That's the "missing/distorted spectrum near channel
edges" symptom.

The fix is to set `decimation=M // 2`, which decimates by `M/2` instead of `M` (2x oversampling). The channel
centers and spacing don't change -- only the rate at which each channel is sampled -- so the same
transition-band content now lands at a resolvable frequency instead of folding onto DC.

Note this does **not** make any single channel's filter more selective (a tone exactly at a crossover
still splits its energy 50/50 between the two adjacent channels, same as without oversampling -- that's a
property of the prototype filter's shape, not the sample rate). What oversampling fixes is *aliasing*
*within* each channel, which is the more serious problem when you want to actually demodulate/use
the signal that lands near an edge rather than just detect that something is there.

In [ ]:
import sys
from pathlib import Path

_src = Path.cwd().parent / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

import numpy as np
import matplotlib.pyplot as plt

from pypoly import PolyphaseAnalysisChannelizer

%matplotlib inline

## Sanity check: channel mapping and output rate

Oversampling must not break the channel-to-frequency mapping fixed in
`01_prototype_filter_and_channel_mapping.ipynb`, and the output should have roughly 2x as many blocks for
the same input length (decimating by `M/2` instead of `M`).

In [ ]:
M = 8
TAPS_PER_CHANNEL = 16

critical = PolyphaseAnalysisChannelizer.from_design(num_channels=M, taps_per_channel=TAPS_PER_CHANNEL)
oversampled = PolyphaseAnalysisChannelizer.from_design(
    num_channels=M, taps_per_channel=TAPS_PER_CHANNEL, decimation=M // 2
)

n = np.arange(8192)
winners = []
for k0 in range(M):
    x = np.exp(1j * 2 * np.pi * k0 * n / M)
    y = oversampled.process(x)
    energy = np.mean(np.abs(y[:, 200:]) ** 2, axis=1)
    winners.append(int(np.argmax(energy)))

print("oversampled channel mapping:", winners)
assert winners == list(range(M))

x = np.ones(257, dtype=complex)
print("critical num blocks:", critical.process(x).shape[1])
print("oversampled num blocks:", oversampled.process(x).shape[1])

## The actual benefit: in-channel aliasing at a crossover frequency

Put a tone exactly halfway between channel 3 and channel 4's centers and look at *channel 3's own output
spectrum* (not which channel has more energy -- both will report similar energy in both implementations).

- Critically sampled: channel 3 is sampled at rate `1/M`, so the true baseband offset (`+0.5` cycles/block)
  aliases to `-0.5` -- exactly the Nyquist fold, i.e. you cannot tell which side of the channel the signal
  is really on.
- Oversampled: channel 3 is sampled at rate `2/M`, so `+0.5` cycles/block (relative to the slower rate)
  resolves to a distinguishable, correct frequency instead of folding onto itself.

In [ ]:
f0 = 3.5 / M  # exact crossover between channel 3 and channel 4
x = np.exp(1j * 2 * np.pi * f0 * n)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for ax, label, channelizer, rate_div in [
    (axes[0], "critically sampled", critical, M),
    (axes[1], "oversampled 2x", oversampled, M // 2),
]:
    y = channelizer.process(x)
    sig = y[3, 200:600]  # channel 3's own output sequence
    spec = np.fft.fftshift(np.fft.fft(sig))
    freqs = np.fft.fftshift(np.fft.fftfreq(len(sig)))
    ax.plot(freqs, 20 * np.log10(np.maximum(np.abs(spec), 1e-6) / np.abs(spec).max()))
    expected = (f0 - 3 / M) * rate_div
    ax.axvline(expected, color="k", linestyle="--", linewidth=1, label=f"expected un-aliased = {expected:.2f}")
    ax.set_title(f"Channel 3 output spectrum\n({label})")
    ax.set_xlabel("cycles / output block")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("magnitude (dB)")
fig.tight_layout()

The critically-sampled plot shows the crossover tone landing at `-0.5` cycles/block -- the Nyquist fold of
the true `+0.5` offset, i.e. it is indistinguishable from a tone on the *other* side of the channel. The
oversampled plot resolves it correctly at the expected, unambiguous frequency. This is the practical
payoff: signal content near a channel edge can actually be demodulated/used in the oversampled case,
whereas in the critically-sampled case it's aliased and unrecoverable.

## How far can you push decimation?

`decimation` can be any integer in `[1, num_channels]`. Smaller values mean more oversampling, but
the default cutoff scaling (`cutoff_ratio = 0.9 * num_channels / decimation`) only stays sane up to a
point: a channel's *farthest* neighbor always sits at exactly `0.5` of Nyquist, regardless of
`num_channels`, so once the auto-scaled cutoff reaches that point the "stopband" at the opposite
channel is literally inside the passband -- no number of taps can fix that.

`design_prototype_filter` now warns (`UserWarning`) when this happens, rather than silently handing
back a non-functional filter. `decimation=1` (maximal oversampling, R=8 for an 8-channel example) is
a clear case of this.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    extreme = PolyphaseAnalysisChannelizer.from_design(
        num_channels=M, taps_per_channel=TAPS_PER_CHANNEL, decimation=1
    )

print(f"warnings raised: {len(caught)}")
if caught:
    print(caught[0].message)

In [ ]:
# And the warning's premise holds: a DC tone leaks into every channel with comparable
# energy instead of concentrating in channel 0 -- there is no real channel separation left.
x_dc = np.ones(8192, dtype=complex)
y_dc = extreme.process(x_dc)
energy_dc = np.mean(np.abs(y_dc[:, 200:]) ** 2, axis=1)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(M), energy_dc)
ax.set_xlabel("channel")
ax.set_ylabel("energy (DC input)")
ax.set_title("decimation=1: DC leaks into every channel equally -- no separation")
ax.grid(True, alpha=0.3)